**End-to-End Recovery Tokens.** This figure charges the complete post-policy autonomous LLM recovery loop: diagnosis, prompts and tool results, planning, validation, and regenerated content. The same fault DAG, model, prompt, tool schema, and commit boundary are used for AgentTX causal rollback, an optimistic temporal checkpoint, and whole-branch abort.

In [ ]:
# ipython -c "%run plot_token_end_to_end.ipynb"

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

STANDARD_WIDTH = 17.8

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

STYLES = {
    'causal': dict(color='#c00000', marker='s', linestyle='-', linewidth=1.0, markersize=3.2),
    'temporal_checkpoint': dict(color='#e78129', marker='x', linestyle=':', linewidth=0.9, markersize=3.6, markeredgewidth=0.9),
    'whole_branch_abort': dict(color='black', marker='o', linestyle='--', linewidth=0.8, markersize=2.8, markerfacecolor='none'),
}
LABELS = {
    'causal': 'AgentTX causal (ours)',
    'temporal_checkpoint': 'optimistic checkpoint',
    'whole_branch_abort': 'whole-branch abort',
}
MODES = list(STYLES)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
result = RESULTS / 'token_end_to_end.csv'
if not result.exists():
    raise FileNotFoundError(f'missing {result}; run experiments/scripts/bench_token_end_to_end.py first')
df = pd.read_csv(result)
numeric = [
    'document_lines', 'total_tokens_mean', 'prompt_tokens_mean',
    'completion_tokens_mean', 'recovery_wall_s_p95', 'success_rate', 'host_leak_rate',
]
df[numeric] = df[numeric].apply(pd.to_numeric)

def series(mode, metric):
    rows = df[df['mode'] == mode].sort_values('document_lines')
    return rows['document_lines'].to_numpy(), rows[metric].to_numpy(dtype=float)

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))
handles = []
panels = [
    ('total_tokens_mean', 'Total API tokens', '(a) Complete recovery cost'),
    ('prompt_tokens_mean', 'Prompt tokens', '(b) Diagnosis and context'),
    ('completion_tokens_mean', 'Completion tokens', '(c) Planning and repair'),
    ('recovery_wall_s_p95', 'Recovery p95 (s)', '(d) End-to-end latency'),
]
for index, (metric, ylabel, subtitle) in enumerate(panels, start=1):
    ax = plt.subplot(2, 2, index)
    for mode in MODES:
        x, y = series(mode, metric)
        handle, = ax.plot(x, y, **STYLES[mode], label=LABELS[mode])
        if index == 1:
            handles.append(handle)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xlabel(f'Document size (# entries)\n{subtitle}', fontsize=7)
    ax.set_xticks(sorted(df['document_lines'].unique()))
    ax.tick_params(axis='both', labelsize=7)
    ax.set_ylim(bottom=-0.03 * max(1.0, ax.get_ylim()[1]))

ax_tokens = fig.axes[0]
x_abort, y_abort = series('whole_branch_abort', 'total_tokens_mean')
x_causal, y_causal = series('causal', 'total_tokens_mean')
saved = y_abort[-1] - y_causal[-1]
ax_tokens.annotate(
    f'{saved:,.0f} tokens saved',
    xy=(x_abort[-1], y_abort[-1]), xytext=(-74, -15), textcoords='offset points',
    fontsize=6.5, color='#c00000',
    arrowprops=dict(arrowstyle='-', color='#c00000', linewidth=0.6),
)

fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.035), ncol=3,
           fontsize=7, columnspacing=1.0, handlelength=1.8, handletextpad=0.35, borderpad=0.3)
plt.tight_layout(pad=0.6, h_pad=1.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.91])
plt.savefig(FIGDIR / 'FIG-Token-End-to-End.pdf', bbox_inches='tight', pad_inches=0.02,
            metadata={'CreationDate': None, 'ModDate': None})
plt.savefig(FIGDIR / 'FIG-Token-End-to-End.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

assert (df['success_rate'] == 1.0).all()
assert (df['host_leak_rate'] == 0.0).all()
